# 05 · Forecast de demanda y agotamiento de stock
**North Star predictiva:** cuántos departamentos absorberá el proyecto en los próximos `N` meses y cuánto demoraría en agotar el stock disponible.

Se modelan dos cosas distintas: **demanda neta de stock** (`separaciones - caídas`) para agotamiento y **ventas/minutas** para conversión comercial.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'replica_cygnus').exists())
sys.path.insert(0, str(ROOT/'src'))
from replica_cygnus.economic_intelligence import install_feature_mart, load_monthly_panel, build_supervised_panel
from replica_cygnus.economic_intelligence.features import select_model_matrix
from replica_cygnus.economic_intelligence.models import backtest_regressors, fit_champion_model
install_feature_mart(); panel = load_monthly_panel(); sup = build_supervised_panel(panel, horizons=(1,3,6,12))
N_MONTHS = 6


## 1. Backtest: demanda neta próxima ventana
La validación deja fuera los meses más recientes. Eso simula la pregunta real: entreno con pasado y predigo futuro.

In [ ]:
target = f'target_mov_neto_next_{N_MONTHS}m'
X, y, feature_cols = select_model_matrix(sup, target, include_macro=True, include_reference=False)
meta = sup.loc[X.index, ['periodo_mes','codigo_proyecto','proyecto']]
scores_demand, fitted_demand, pred_demand = backtest_regressors(X, y, meta, nonnegative_target=False, test_months=6)
scores_demand

## 2. Backtest: minutas próximas ventana
Minutas son conteos no negativos, por lo que el challenger incluye Poisson además de Ridge, Random Forest y Gradient Boosting.

In [ ]:
target_min = f'target_minutas_next_{N_MONTHS}m'
Xm, ym, feature_cols_m = select_model_matrix(sup, target_min, include_macro=True, include_reference=False)
meta_m = sup.loc[Xm.index, ['periodo_mes','codigo_proyecto','proyecto']]
scores_min, fitted_min, pred_min = backtest_regressors(Xm, ym, meta_m, nonnegative_target=True, test_months=6)
scores_min

## 3. Champion y predicción para cada proyecto
El champion se elige por MAE de holdout. El forecast de `N` meses es directo; evita encadenar 6 errores mensuales para la métrica principal.

In [ ]:
champion_name = scores_demand.iloc[0]['name']
champion_min_name = scores_min.iloc[0]['name']
champion = fit_champion_model(X, y, model_name=champion_name)
champion_min = fit_champion_model(Xm, ym, model_name=champion_min_name, nonnegative_target=True)
latest = sup.sort_values('periodo_mes').groupby('codigo_proyecto', as_index=False).tail(1).copy()
X_latest = latest[feature_cols].replace([np.inf,-np.inf], np.nan)
Xm_latest = latest[feature_cols_m].replace([np.inf,-np.inf], np.nan)
latest[f'forecast_demanda_neta_{N_MONTHS}m'] = np.clip(champion.predict(X_latest), 0, None)
latest[f'forecast_minutas_{N_MONTHS}m'] = np.clip(champion_min.predict(Xm_latest), 0, None)
latest['ritmo_mensual_pred'] = latest[f'forecast_demanda_neta_{N_MONTHS}m'] / N_MONTHS
latest['meses_a_agotar_stock_p50'] = latest['saldo_final_observado'] / latest['ritmo_mensual_pred'].replace(0, np.nan)
latest['mes_stockout_p50'] = latest.apply(lambda r: (pd.Timestamp(r['periodo_mes']).to_period('M') + int(np.ceil(r['meses_a_agotar_stock_p50']))).to_timestamp() if pd.notna(r['meses_a_agotar_stock_p50']) else pd.NaT, axis=1)
forecast = latest[['codigo_proyecto','proyecto','periodo_mes','saldo_final_observado',f'forecast_demanda_neta_{N_MONTHS}m',f'forecast_minutas_{N_MONTHS}m','ritmo_mensual_pred','meses_a_agotar_stock_p50','mes_stockout_p50']]
forecast.sort_values('meses_a_agotar_stock_p50')

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
plot = forecast.sort_values('meses_a_agotar_stock_p50')
ax.bar(plot['proyecto'], plot['meses_a_agotar_stock_p50'])
ax.set_ylabel('Meses estimados'); ax.set_title(f'Agotamiento de stock a ritmo predictivo · horizonte {N_MONTHS}m'); ax.tick_params(axis='x', rotation=45); plt.show()

## 4. Qué falta para un forecast de producción
- intervalos P10/P50/P90 mediante bootstrap o modelos cuantílicos;
- snapshots históricos de precio/descuento;
- macro externo con rezagos;
- backtest por proyecto y por etapa;
- drift y champion/challenger;
- reconciliación: demanda neta pronosticada no puede absorber más stock que el disponible.

**Regla:** para agotamiento usamos demanda neta de stock. Las minutas se pronostican en paralelo porque miden otra fase del funnel.